## Trabajo de transformaciones de Excel, Orange y Python

**Integrantes:** Miguel Vangeas Jose Vanegas  
**Institucion:** Universidad Politecnica Salesiana  
**Fecha:** 25/04/2026

### Introduccion

En esta practica se trabajo con un conjunto de datos de clientes que contiene variables como sexo, edad, pais y nivel de satisfaccion. El objetivo principal fue preparar la informacion para que quede lista para su analisis y para futuros modelos de aprendizaje automatico.

Durante el proceso se aplicaron transformaciones sobre variables categoricas y numericas. Primero se aplico One-Hot Encoding a las variables nominales y codificacion ordinal a la variable de satisfaccion; despues, todo el conjunto se estandarizo con StandardScaler para obtener una version numerica homogena.

Este trabajo permite ver, de forma practica, como cambia un dataset cuando se limpia y transforma correctamente antes de ser usado en un analisis posterior.

## 1. Preparacion de datos

### 1.1 Importacion de librerias necesarias

En esta primera parte se cargan las librerias que se usaran durante todo el informe. `numpy` y `pandas` permiten trabajar con los datos, `copy` sirve para crear una copia segura del dataset original y las herramientas de `sklearn` se utilizan para transformar variables y construir el pipeline de preprocesamiento.

In [9]:
import numpy as np
import pandas as pd
import copy
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print('Librerias cargadas')

Librerias cargadas


### 1.2 Cargar datos y copia de seguridad

Aqui se lee el archivo Excel que contiene la informacion de los clientes. Despues de cargarlo se revisa la forma del dataset y se hace una copia para trabajar sin modificar el archivo original.

Tambien se normaliza el texto de las columnas categoricas para evitar errores por espacios o mayusculas.

In [12]:
# 1. Carga de datos
archivo_excel = 'Libro1.xlsx'
df_original = pd.read_excel(archivo_excel, sheet_name='Hoja1')

# --- LA TABLA ORIGINAL ---
print("********** 1. TABLA ORIGINAL (Antes de procesar) **********")
print(df_original.head(6).to_string(index=False))
print("\n" + "="*60 + "\n")

# 2. Creamos copia y limpiamos
df_procesado = copy.deepcopy(df_original)
df_procesado['pais'] = df_procesado['pais'].astype(str).str.strip().str.lower()
df_procesado['NivelSatifaccion'] = df_procesado['NivelSatifaccion'].astype(str).str.strip().str.lower()

********** 1. TABLA ORIGINAL (Antes de procesar) **********
Sexo  edad     pais NivelSatifaccion
   F    65   Brasil        me gusta 
   M    26   España      no me gusta
   F    21    Chile          neutral
   M    12  Ecuador      no me gusta
   F    32   España        me gusta 
   M     9 Ecuador       no me gusta




### 1.3 Definicion de variables estructurales

En este punto se separan las columnas segun el tipo de informacion que contienen. Las variables nominales no tienen un orden natural, las numericas representan cantidades y las ordinales si tienen una secuencia logica.

Esta clasificacion es importante porque cada tipo de variable necesita un tratamiento diferente antes de entrar al modelo.

In [13]:
# Definición de variables
vars_nominales = ['Sexo', 'pais']
vars_numericas = ['edad']
vars_ordinales = ['NivelSatifaccion']
mi_orden_logico = [['no me gusta', 'neutral', 'me gusta']]

print('Variables definidas: ', vars_nominales, vars_numericas, vars_ordinales)

Variables definidas:  ['Sexo', 'pais'] ['edad'] ['NivelSatifaccion']


### 1.4 Construccion del preprocesador

En esta seccion solo se define la estructura de transformacion que usara el script. Primero se preparan las reglas para las variables nominales y ordinales, y mas adelante se aplicara una estandarizacion total sobre el resultado.

Separar esta parte ayuda a entender mejor que aqui solo se arma el preprocesador, sin ejecutar todavia el procesamiento completo del dataset.

In [16]:
# 3. PREPROCESADOR Y PIPELINE (Estandarización Total)
preprocesador_base = ColumnTransformer(transformers=[
    ('cat_nom', OneHotEncoder(sparse_output=False, handle_unknown="ignore"), vars_nominales),
    ('cat_ord', OrdinalEncoder(categories=mi_orden_logico), vars_ordinales)
], remainder='passthrough')

pipe_maestro = Pipeline(steps=[
    ('preprocesamiento', preprocesador_base),
    ('estandarizacion_total', StandardScaler()) # Estandariza todo de golpe
])

### 1.5 Ejecucion y reconstruccion del resultado, Exportacion y verificacion de resultados

En esta parte se aplica el pipeline ya definido al dataset procesado. Luego se reconstruye la tabla con los nombres de columnas correspondientes y se calculan estadisticas de comprobacion para validar la estandarizacion.

In [17]:
# 4. EJECUCIÓN DEL PIPELINE
X_transformado = pipe_maestro.fit_transform(df_procesado)

# 5. RECONSTRUCCIÓN DE LA TABLA
nombres_nominales = pipe_maestro.named_steps['preprocesamiento'].named_transformers_['cat_nom'].get_feature_names_out(vars_nominales)
columnas_finales = list(nombres_nominales) + vars_ordinales + vars_numericas

df_final = pd.DataFrame(data=X_transformado, columns=columnas_finales)

# --- IMPRESIÓN 2: LA TABLA ESTANDARIZADA ---
print("********** 2. TABLA ESTANDARIZADA (Todas las variables) **********")
print(df_final.round(4).head(6).to_string(index=False))
print("\n" + "="*60 + "\n")

# --- IMPRESIÓN 3: TABLA DE COMPROBACIÓN ESTADÍSTICA ---
print("********** 3. ESTADÍSTICAS (Media y Desviación Estándar) **********")
# Calculamos las estadísticas
df_estadisticas = pd.DataFrame({
    'Media (Promedio)': df_final.mean(),
    'Desviación Estándar': df_final.std(ddof=0) # ddof=0 es la fórmula poblacional que usa StandardScaler
})

# Le ponemos round(2) para redondear y que se vean los "0.00" perfectos 
# (sino la computadora a veces imprime notación científica como 3.7e-17)
print(df_estadisticas.round(2).to_string())

********** 2. TABLA ESTANDARIZADA (Todas las variables) **********
 Sexo_F  Sexo_M  pais_brasil  pais_chile  pais_ecuador  pais_españa  NivelSatifaccion    edad
    1.0    -1.0       2.2361     -0.4472       -0.7071      -0.7071            1.2999  2.0270
   -1.0     1.0      -0.4472     -0.4472       -0.7071       1.4142           -0.9285 -0.0811
    1.0    -1.0      -0.4472      2.2361       -0.7071      -0.7071            0.1857 -0.3514
   -1.0     1.0      -0.4472     -0.4472        1.4142      -0.7071           -0.9285 -0.8378
    1.0    -1.0      -0.4472     -0.4472       -0.7071       1.4142            1.2999  0.2432
   -1.0     1.0      -0.4472     -0.4472        1.4142      -0.7071           -0.9285 -1.0000


********** 3. ESTADÍSTICAS (Media y Desviación Estándar) **********
                  Media (Promedio)  Desviación Estándar
Sexo_F                         0.0                  1.0
Sexo_M                         0.0                  1.0
pais_brasil                    0.0   

### 1.6 Exportacion y verificacion de resultados

Por ultimo, el dataset transformado se guarda en un nuevo archivo Excel para no sobrescribir el original. El archivo generado es `Dataset_Full_Estandarizado.xlsx`, y la vista previa permite confirmar que los datos quedaron transformados y listos para usarse en el siguiente paso del analisis.

In [18]:
# Guardamos el archivo
df_final.to_excel('Dataset_Full_Estandarizado.xlsx', index=False)

### Conclusión

Esta practica permitio consolidar un flujo de trabajo completo para la preparacion de datos, transformando un archivo crudo en un dataset numerico listo para analisis o para futuros algoritmos de aprendizaje automatico. Durante el desarrollo se vio que la limpieza previa, como la normalizacion de texto, es clave para evitar errores de ejecucion y obtener columnas coherentes.

Tambien se comprobo que sincronizar correctamente el entorno de Python y sus librerias es fundamental para que el notebook funcione sin problemas. La implementacion del pipeline, combinando codificacion ordinal, One-Hot Encoding y estandarizacion, demostro que estas transformaciones se pueden automatizar de forma ordenada y reproducible. Como mejora futura, se recomienda agregar imputacion de valores faltantes y documentar las versiones de las dependencias para facilitar la reproduccion del trabajo.